In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
MODEL_FILE='model.pkl'
PIPELINE_FILE='pipeline.pkl'
def build_pipeline(num_attribs, cat_attribs):
     num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
     ])
     cat_pipeline = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
     ])
     full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs)
     ])
     return full_pipeline



In [6]:
if not os.path.exists(MODEL_FILE):
# TRAINING PHASE
    laptop = pd.read_csv("laptop_prices.csv")
    laptop = laptop.drop(columns=["Product", "CPU_model", "GPU_model"])

    
    laptop['Price_cat'] = pd.cut(laptop["Price_euros"],bins=[0, 500, 1000, 1500, 2000,3000, np.inf],labels=[1, 2, 3, 4, 5,6])
    
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    
    # for train_index, _ in split.split(housing, housing['income_cat']):
    #      housing = housing.loc[train_index].drop("income_cat", axis=1)

    
    for train_index,test_index in split.split(laptop,laptop['Price_cat']):
        laptop.loc[test_index].to_csv("test_actual.csv", index=False)
        
        laptop.loc[test_index] \
              .drop(["Price_cat", "Price_euros"], axis=1) \
              .to_csv("input.csv", index=False)
        # laptop.loc[test_index].drop("Price_cat", axis=1).to_csv("input.csv",index=False)
        laptop= laptop.loc[train_index].drop("Price_cat", axis=1)
        
                
   
    laptop_labels = laptop["Price_euros"].copy()
    laptop_features = laptop.drop("Price_euros", axis=1)
    
    num_attribs = laptop_features.drop(columns=['Company','TypeName','OS','Screen','Touchscreen'
                                            ,'IPSpanel','RetinaDisplay','CPU_company','PrimaryStorageType'
                                            ,'SecondaryStorageType','GPU_company'], axis=1).columns.tolist()

    cat_attribs = [
    'Company','TypeName','OS','Screen','Touchscreen',
    'IPSpanel','RetinaDisplay','CPU_company',
    'PrimaryStorageType','SecondaryStorageType',
    'GPU_company']


    
    pipeline  = build_pipeline(num_attribs, cat_attribs)
    laptop_prepared = pipeline.fit_transform(laptop_features)

    # model =Ridge(alpha=1.0,random_state=42)
    # model.fit(laptop_prepared, laptop_labels)
    model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1)

    model.fit(laptop_prepared, laptop_labels)


    # model = RandomForestRegressor(random_state=42)
    # model.fit(laptop_prepared, laptop_labels)

    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline , PIPELINE_FILE)

    print("Model trained and saved.")
else:
     model = joblib.load(MODEL_FILE)
     pipeline = joblib.load(PIPELINE_FILE)
    
     input_data = pd.read_csv("input.csv")
     transformed_input = pipeline.transform(input_data)
    
     predictions = model.predict(transformed_input)
     # input_data["Price_euros"] = predictions
     input_data["Predicted_Price_euros"] = predictions

   
     input_data.to_csv("output.csv", index=False)
     print("Inference complete. Results saved to output.csv")


Inference complete. Results saved to output.csv
